In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical

class Router(nn.Module):
    def __init__(self, L):
        super().__init__()
        self.L = L
        self.phi = nn.Parameter(torch.zeros(L))

    def forward(self):
        """Return categorical probabilities over L positions."""
        return torch.softmax(self.phi, dim=-1)

    def sample_mask(self, p=None, num_sample=1, batch_size=1):
        """
        Sample `num_sample` positions for each batch element.

        Args:
            p (torch.Tensor, optional): Probabilities over L positions. Defaults to self.forward().
            num_sample (int): Number of positions to sample.
            unique (bool): If True, samples without replacement.
            batch_size (int): Number of independent draws (batches).

        Returns:
            torch.LongTensor of shape (batch_size, num_sample)
                if num_sample > 1, else (batch_size,)
        """
        if p is None:
            p = self.forward()  # shape (L,)
        else:
            p = torch.as_tensor(p, dtype=torch.float32, device=self.phi.device)

        if num_sample > 1:
            # Without replacement
            idx = torch.stack([torch.multinomial(p, num_samples=num_sample, replacement=False)
                               for _ in range(batch_size)])
        else:
            # With replacement (independent samples)
            dist = Categorical(p)
            if num_sample == 1:
                idx = dist.sample((batch_size,))  # (batch_size,)
            else:
                idx = dist.sample((batch_size, num_sample))  # (batch_size, num_sample)

        return idx

    def log_prob(self, idx, reduce: str = "none"):
        """
        Compute log-probabilities under current policy.

        Args:
            idx: Tensor of indices (batch_size,) or (batch_size, num_sample)
            reduce: "none" | "sum" | "mean"
        """
        p = self.forward()
        logp = torch.log(p.clamp(min=1e-8))
        idx = torch.as_tensor(idx, dtype=torch.long, device=logp.device)
        gathered = logp.index_select(0, idx.reshape(-1)).reshape(idx.shape)

        if reduce == "none":
            return gathered
        elif reduce == "sum":
            return gathered.sum(dim=-1)
        elif reduce == "mean":
            return gathered.mean(dim=-1)
        else:
            raise ValueError(f"Invalid reduce='{reduce}'. Use 'none' | 'sum' | 'mean'.")

    def entropy(self, p: torch.Tensor = None) -> torch.Tensor:
        """Entropy H(p) in nats."""
        if p is None:
            p = self.forward()
        p_safe = p.clamp(min=1e-12)
        return -(p_safe * torch.log(p_safe)).sum()

In [ ]:
pi = Router(10)

In [ ]:
pi.sample_mask(num_sample=2, batch_size=4)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Example: 10 timesteps × 5 categories
np.random.seed(42)
data = np.random.dirichlet(np.ones(5), size=10)

plt.figure(figsize=(8, 5))
sns.heatmap(data, annot=True, cmap='Blues', cbar=True,
            xticklabels=[f'Cat {i}' for i in range(1, 6)],
            yticklabels=[f'Time {t}' for t in range(1, 11)])
plt.title('Categorical Distributions over 10 Timesteps')
plt.xlabel('Categories')
plt.ylabel('Timesteps')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# same data as before
timesteps = np.arange(1, 11)
for cat in range(5):
    plt.plot(timesteps, data[:, cat], marker='o', label=f'Cat {cat+1}')

plt.title('Category Probability Across Timesteps')
plt.xlabel('Timestep')
plt.ylabel('Probability')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.stackplot(timesteps, data.T, labels=[f'Cat {i+1}' for i in range(5)])
plt.title('Evolution of Categorical Distribution Over Time')
plt.xlabel('Timestep')
plt.ylabel('Probability')
plt.legend(loc='upper left')
plt.show()
